# Adhan PyTorch/Colab Workspace — Scaffold Only

**Scope of this notebook:** environment validation, corpus mount, and loading one real batch.
**No model code, no training loop** — that starts in a follow-up notebook once this scaffold is
proven working.

**Done when:** the batch cell below prints `input_ids` and `targets` with shapes
`(batch_size, seq_len - 1)` and a dtype of `int64`, loaded from a real `train.bin` produced by
`scripts/prepare_slm_corpus.py`, on a free Colab runtime.

This reuses the repo's own code (cloned fresh below) rather than duplicating logic here:
`src/adhan_slm/data/packing.py`'s `open_shard_memmap()` — the single manifest-driven reader for
shards that `prepare_slm_corpus.py` writes.


## 1. Clone the repo (idempotent)

Gives us `src/adhan_slm/` without duplicating any of its code into this
notebook — if it changes on `main`, re-running this cell picks up the
change.

In [1]:
import os

REPO_DIR = "/content/adhan"

if os.path.isdir(REPO_DIR):
    print("Repo already present, pulling latest ...")
    !cd {REPO_DIR} && git pull
else:
    print("Cloning adhan (shallow) ...")
    !git clone --depth 1 https://github.com/yazhi-lem/adhan.git {REPO_DIR}


Cloning adhan (shallow) ...
Cloning into '/content/adhan'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 189 (delta 4), reused 124 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (189/189), 651.61 KiB | 899.00 KiB/s, done.
Resolving deltas: 100% (4/4), done.


## 2. Environment check (validated against a committed file)

Colab's free runtime already ships `torch` matched to whatever GPU/CUDA driver it handed you for
this session — force-reinstalling a different pinned `torch` risks breaking that match, and a
forced `numpy` downgrade mid-session corrupts the runtime in a way even a kernel restart does not
fix.

So this does not install anything. It validates what is actually present against
`notebooks/colab_constraints.txt` (committed in this repo), prints PASS/WARN per package, and
writes the full `pip freeze` for the session. If something is out of bounds you get a clear warning
and the exact version, instead of a silent failure much later.


In [ ]:
import re
import subprocess
import sys
from pathlib import Path

CONSTRAINTS = Path(REPO_DIR) / "notebooks" / "colab_constraints.txt"
print("Python:", sys.version.split()[0])

import torch
import numpy as np

print(f"torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"numpy: {np.__version__}")


def check_constraints(path: Path) -> int:
    """Validate installed versions against the committed bounds. Never installs anything."""
    installed = {
        "python": sys.version.split()[0],
        "torch": torch.__version__.split("+")[0],  # drop the +cuXXX local version
        "numpy": np.__version__,
    }
    problems = 0
    for raw in path.read_text().splitlines():
        line = raw.split("#", 1)[0].strip()
        if not line:
            continue
        name = re.split(r"[<>=!]", line, maxsplit=1)[0].strip()
        spec = line[len(name) :]
        have = installed.get(name)
        if have is None:
            print(f"  ?  {name}: not imported here, cannot validate")
            continue
        # A tiny comparator: every bound must hold. `packaging` may not be present on Colab.
        ok = True
        for op, want in re.findall(r"(==|>=|<=|>|<)\s*([0-9][0-9.]*)", spec):
            a = tuple(int(x) for x in have.split(".")[: len(want.split("."))])
            b = tuple(int(x) for x in want.split("."))
            ok &= {
                "==": a == b,
                ">=": a >= b,
                "<=": a <= b,
                ">": a > b,
                "<": a < b,
            }[op]
        print(f"  {'PASS' if ok else 'WARN'}  {name}: {have} (constraint '{name}{spec}')")
        problems += 0 if ok else 1
    return problems


problems = check_constraints(CONSTRAINTS)
if problems:
    print(
        f"\nWARNING: {problems} package(s) outside notebooks/colab_constraints.txt.\n"
        "Do NOT pip-install over Colab's CUDA-matched torch; either accept the warning for a\n"
        "smoke test or move to a runtime that ships the supported stack."
    )

# Full session snapshot, for reproducing a run. Ephemeral: /content is not committed.
with open("/content/environment-lock.txt", "w") as fh:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=fh, check=False)
print("\nFull environment snapshot written to /content/environment-lock.txt")


## 3. Mount the corpus

`data/` is gitignored in the repo (raw corpora are never committed), so
`train.bin` / `train.bin.manifest.json` / `vocab.json` / `merges.txt`
(all produced by `python scripts/prepare_slm_corpus.py` — see that
script, not reinvented here) need to already exist somewhere reachable
from Colab:

- **Preferred:** upload the `prepare_slm_corpus.py` output directory to
  Google Drive once, then mount Drive here on every session.
- **Fallback:** no Drive copy yet? The next cell falls back to a direct
  file upload picker for the 4 files above.

Either way, set `CORPUS_DIR` below to wherever the 4 files end up.

In [3]:
from pathlib import Path

CORPUS_DIR = Path("/content/drive/MyDrive/adhan/data/final/tamil_slm")  # <-- adjust to your Drive path
REQUIRED_FILES = ["train.bin", "train.bin.manifest.json", "vocab.json", "merges.txt"]

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
except ImportError:
    print("Not running in Colab (or Drive unavailable) -- skipping mount.")

if not CORPUS_DIR.exists() or not all((CORPUS_DIR / f).exists() for f in REQUIRED_FILES):
    print(f"Corpus not found at {CORPUS_DIR} -- falling back to direct upload.")
    print(f"Please select all 4 files: {REQUIRED_FILES}")
    from google.colab import files

    CORPUS_DIR = Path("/content/uploaded_corpus")
    CORPUS_DIR.mkdir(exist_ok=True)
    uploaded = files.upload()
    for name, data in uploaded.items():
        (CORPUS_DIR / name).write_bytes(data)

missing = [f for f in REQUIRED_FILES if not (CORPUS_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Still missing {missing} in {CORPUS_DIR}. "
        "Generate them first with: python scripts/prepare_slm_corpus.py "
        "--corpus <your_corpus_dir> --out data/final/tamil_slm"
    )
print(f"Corpus ready at {CORPUS_DIR}: {REQUIRED_FILES}")


Mounted at /content/drive
Corpus ready at /content/drive/MyDrive/adhan/data/final/tamil_slm: ['train.bin', 'train.bin.manifest.json', 'vocab.json', 'merges.txt']


## 4. Load one real batch (memory-mapped)

Uses `open_shard_memmap()` from `src/adhan_slm/data/packing.py`: the one reader every trainer
should use, because row width, dtype and row count all come from the `*.manifest.json` sidecar
rather than from guessing at the file size.

`open_shard_memmap` memory-maps the shard, so `shard[:BATCH_SIZE]` touches only those rows. An
earlier draft used `read_shard()`, which calls `numpy.fromfile` and therefore materialises the
**entire** corpus in RAM before slicing eight rows — fine on a 3k-token toy shard, fatal on the
larger shards this notebook exists to mount.

The causal-LM split is the same one `MemmapShardDataset.get_batch()` uses: `row[:-1]` is the
input, `row[1:]` is the target. (A previous version of this notebook claimed
`MemmapShardDataset` expected a different on-disk format; that was a real bug, and it is fixed —
see #79.)


In [ ]:
import sys

sys.path.insert(0, f"{REPO_DIR}/src")

import numpy as np

from adhan_slm.data.packing import load_manifest, open_shard_memmap  # noqa: E402

SHARD = CORPUS_DIR / "train.bin"
manifest = load_manifest(SHARD)
print(
    f"manifest: seq_len={manifest.seq_len}, n_sequences={manifest.n_sequences}, "
    f"vocab_size={manifest.vocab_size}, dtype={manifest.dtype}"
)

# Memory-mapped: only the rows we index are read, so this stays flat in RAM for any shard size.
shard = open_shard_memmap(SHARD, expected_seq_len=manifest.seq_len)
print(f"mapped shard shape: {shard.shape}  (n_sequences, seq_len)")

BATCH_SIZE = 8
rows = shard[:BATCH_SIZE]                      # reads BATCH_SIZE rows, not the whole file
input_ids = rows[:, :-1].astype(np.int64)
targets = rows[:, 1:].astype(np.int64)

# The sequence-length contract every downstream trainer relies on.
expected = (min(BATCH_SIZE, manifest.n_sequences), manifest.seq_len - 1)
assert input_ids.shape == expected, f"input_ids {input_ids.shape} != {expected}"
assert targets.shape == expected, f"targets {targets.shape} != {expected}"
assert input_ids.dtype == np.int64 and targets.dtype == np.int64, "both must be int64"

print(f"input_ids: {input_ids.shape} {input_ids.dtype}")
print(f"targets:   {targets.shape} {targets.dtype}")
print(f"token id range: {int(input_ids.min())} .. {int(input_ids.max())} "
      f"(vocab_size={manifest.vocab_size})")
assert input_ids.max() < manifest.vocab_size, "token id outside the vocabulary"


## 5. Sanity check: decode one sequence back to Tamil

Not model code -- just proves the mounted corpus round-trips correctly
(right vocab, right byte order) rather than being garbled or
misaligned, before anyone builds a training loop on top of it.

In [5]:
from adhan_slm.tokenizer import SwaramTokenizer  # noqa: E402

tok = SwaramTokenizer.from_files(str(CORPUS_DIR / "vocab.json"), str(CORPUS_DIR / "merges.txt"))
decoded = tok.decode(batch[0].tolist())
print("Decoded first sequence in the batch:")
print(decoded)


Decoded first sequence in the batch:
திருக்குறள் 579:
ஒறுத்தாற்றும் பண்பினார் கண்ணும்கண் ணோடிப்
பொறுத்தாற்றும் பண்பே தலை.

மு. வரதராசனார் உரை: தண்டித்தற்குரிய தன்மை உடையவரிடத்திலும் கண்ணோட்டம் செய்து ( அவர் செய்த குற்றத்தைப்) பொருத்துக் காக்கும் பண்பே சிறந்தது.

சாலமன் பாப்பையா உரை: தம்மை வருத்தும் இயல்புடையவரிடத்திலும் கண்ணோட்டம் கொண்டு, அவர்தம் பிழையைப் பொறுக்கும் பண்பே சிறந்தது.

கலைஞர் உரை: அழிக்க நினைத்திடும் இயல்புடையவரிடத்திலும் பொறுமை காட்டுவது மிக உயர்ந்த பண்பாகும்திருக்குறள் 281:
எள்ளாமை


## Next steps (out of scope for this notebook)

Environment, corpus mount, batch loading and the `(B, seq_len - 1)` split are proven above, on the
reader that is correct for shards `prepare_slm_corpus.py` writes.

Model definition and the training loop are the next task, built on top of this scaffold rather than
inside it. `MemmapShardDataset` in `scripts/train_efficient.py` already reads this same format
(fixed in #79) and yields the same `row[:-1]` / `row[1:]` split, so the training notebook can use
it directly instead of hand-rolling a loader.
